# 159.224 Week 9 Tutorial: Self Attention Calculation

## Main question for this week

> How does **one token** attend to the other tokens?

This notebook follows the same notation as the lecture slides.

* each token vector $x_i$ is a **column vector**;
* each query $q_i$, key $k_i$, and value $v_i$ is also a **column vector**;
* the query key dot product is therefore written as $q_i^\top k_j$.

The goal this week is to understand the calculation from the perspective of **one token**.


## What you need to know

By the end of Week 9, you should be comfortable with these five ideas.

1. Every token produces its own query, key, and value:

$$
q_i=W_Q^\top x_i,
\qquad
k_i=W_K^\top x_i,
\qquad
v_i=W_V^\top x_i.
$$

2. For token $i$, its query is compared with every key $j$. Here, $s_{ij}$ is the **relevance score of token $i$ on token $j$**: it measures how relevant key $k_j$ is to query $q_i$.

$$
s_{ij}
=
\frac{q_i^\top k_j}{\sqrt{d_k}}.
$$
        

3. Softmax is applied to the scores for one fixed query:

$$
a_{ij}
=
\frac{\exp(s_{ij})}
{\sum_{m=1}^{n}\exp(s_{im})}.
$$

4. The attention weights for one query $q_i$ sum to 1:

$$
\sum_{j=1}^{n}a_{ij}=1.
$$

5. The new representation of token $i$ is a weighted sum of the values:

$$
z_i
=
\sum_{j=1}^{n}a_{ij}v_j.
$$

Note that the full matrix calculation is introduced in Week 10. Not the focus this week. This week we stay with the **one token perspective**.


## Setup

We only need PyTorch and Python's `math` module.

If you are using Google Colab, the code below should run directly.


In [5]:
import math
import torch
import torch.nn as nn

torch.manual_seed(0)

print("PyTorch version:", torch.__version__)


PyTorch version: 2.11.0+cpu


# Part 1: Query, Key, and Value

Let the input sequence be:

$$
x_1,x_2,\ldots,x_n.
$$

Following the lecture slides, each $x_i$ is a **column vector**:

$$
x_i\in\mathbb{R}^{d_{\mathrm{model}}}.
$$

Each token is projected in three ways:

$$
q_i=W_Q^\top x_i,
\qquad
k_i=W_K^\top x_i,
\qquad
v_i=W_V^\top x_i.
$$

with:

$$
W_Q,W_K\in\mathbb{R}^{d_{\mathrm{model}}\times d_k},
\qquad
W_V\in\mathbb{R}^{d_{\mathrm{model}}\times d_v}.
$$

Therefore:

$$
q_i,k_i\in\mathbb{R}^{d_k},
\qquad
v_i\in\mathbb{R}^{d_v}.
$$

For token $i$:

* **Query** $q_i$:  what token $i$ is looking for.
* **Key** $k_j$:  how token $j$ can be matched.
* **Value** $v_j$:  the information that can be taken from token $j$.

The key idea is:

> **Queries and keys are used for matching. Values are used for the output.**


### Quick concept check

Which statement is correct?

**A.** The query is compared with the values, and the keys are combined to produce the output.  
**B.** The query is compared with the keys, and the values are combined using the resulting weights.  
**C.** The query, keys, and values are all averaged before softmax.  
**D.** Only the keys are needed to calculate the final output.

<details>
<summary><b>Check answer</b></summary>

**Answer: B.**

The query and keys are used to calculate relevance scores.

The resulting attention weights are then used to combine the values.

</details>


# Part 2: The entire calculation for one token

Stand at token $i$.

Its query $q_i$ is compared with every key $k_j$.

## Step 1: Query key matching and scaling

For every token $j$:

$$
s_{ij}
=
\frac{q_i^\top k_j}{\sqrt{d_k}}.
$$

Because $q_i$ and $k_j$ are column vectors:

$$
q_i^\top:(1\times d_k),
\qquad
k_j:(d_k\times1),
$$

so:

$$
q_i^\top k_j
$$

is one scalar score.


## Step 2: Softmax

For one fixed query $i$:

$$
a_{ij}
=
\frac{\exp(s_{ij})}
{\sum_{m=1}^{n}\exp(s_{im})}.
$$

The weights satisfy:

$$
\sum_{j=1}^{n}a_{ij}=1.
$$

## Step 3: Weighted sum of the values

The attention output for token $i$ is:

$$
z_i
=
\sum_{j=1}^{n}a_{ij}v_j.
$$

This is the complete self attention calculation from one token's perspective.

# Part 3: A tiny numeric example

Now we will calculate self attention for **one query token** using very small numbers.

We use:

$$
n=3,
\qquad
d_k=2,
\qquad
d_v=3.
$$

Following the lecture notation:

$$
q_i,k_j\in\mathbb{R}^{d_k\times1},
\qquad
v_j\in\mathbb{R}^{d_v\times1}.
$$

So $q_i$, every $k_j$, and every $v_j$ are still **column vectors**.

The purpose of this example is only to make the calculation easy to see.


## Step 1: Create one query and all keys

Let the query of token $i$ be:

$$
q_i
=
\begin{bmatrix}
1\\
0
\end{bmatrix}
\in
\mathbb{R}^{2\times1}.
$$

We have three key vectors:

$$
k_1
=
\begin{bmatrix}
1\\
0
\end{bmatrix},
\qquad
k_2
=
\begin{bmatrix}
0\\
1
\end{bmatrix},
\qquad
k_3
=
\begin{bmatrix}
1\\
1
\end{bmatrix}.
$$

Each key is a column vector with shape:

$$
(2\times1).
$$

For this small example, we simply place the three key vectors **side by side**:

$$
K_{\mathrm{cols}}
=
\begin{bmatrix}
| & | & |\\
k_1 & k_2 & k_3\\
| & | & |
\end{bmatrix}
=
\begin{bmatrix}
1 & 0 & 1\\
0 & 1 & 1
\end{bmatrix}
\in
\mathbb{R}^{2\times3}.
$$

So:

* column 1 is $k_1$
* column 2 is $k_2$
* column 3 is $k_3$

### Important

$K_{\mathrm{cols}}$ is only a convenient helper for this Week 9 example.

It is **not** the formal matrix $K$ used in the Week 10 matrix form.

In Week 10, we will introduce the formal matrices $Q$, $K$, and $V$ exactly as defined in the lecture slides.


In [6]:
# q_i is a column vector: shape (d_k, 1)
q_i = torch.tensor([
    [1.0],
    [0.0]
])

# Each column is one key vector k_j.
# Shape: (d_k, n) = (2, 3)
keys_as_columns = torch.tensor([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0]
])

print("q_i shape:            ", q_i.shape)
print("keys_as_columns shape:", keys_as_columns.shape)


q_i shape:             torch.Size([2, 1])
keys_as_columns shape: torch.Size([2, 3])


## Step 2: Calculate the Query and Key dot products

For one pair of tokens, the lecture formula is:

$$
q_i^\top k_j.
$$

Because:

$$
q_i
\in
\mathbb{R}^{d_k\times1},
$$

we have:

$$
q_i^\top
\in
\mathbb{R}^{1\times d_k}.
$$

For our example:

$$
q_i^\top
=
\begin{bmatrix}
1 & 0
\end{bmatrix}
\in
\mathbb{R}^{1\times2}.
$$

The three keys are already stored as columns:

$$
K_{\mathrm{cols}}
=
\begin{bmatrix}
k_1 & k_2 & k_3
\end{bmatrix}
\in
\mathbb{R}^{2\times3}.
$$

Therefore we can compare the **same query with all three keys at once**:

$$
q_i^\top K_{\mathrm{cols}}
=
\begin{bmatrix}
1 & 0
\end{bmatrix}
\begin{bmatrix}
1 & 0 & 1\\
0 & 1 & 1
\end{bmatrix}.
$$

The result is:

$$
\begin{bmatrix}
q_i^\top k_1 &
q_i^\top k_2 &
q_i^\top k_3
\end{bmatrix}.
$$

The shape calculation is:

$$
(1\times2)(2\times3)
\rightarrow
(1\times3).
$$

So one query produces three scores, one for each key.

In code, this is simply:

```python
raw_scores = q_i.T @ keys_as_columns
```


In [7]:
# q_i.T has shape (1, d_k).
# keys_as_columns has shape (d_k, n).
# The result contains q_i^T k_1, q_i^T k_2, ..., q_i^T k_n.
raw_scores = q_i.T @ keys_as_columns

print("raw scores:", raw_scores)
print("shape:     ", raw_scores.shape)


raw scores: tensor([[1., 0., 1.]])
shape:      torch.Size([1, 3])


## Step 3: Scale by $\sqrt{d_k}$

The lecture defines the scaled relevance score as:

$$
s_{ij}
=
\frac{q_i^\top k_j}{\sqrt{d_k}}.
$$

Here:

$$
d_k=2.
$$

Therefore, every raw dot product is divided by:

$$
\sqrt{2}.
$$

The calculation is:

$$
\begin{bmatrix}
s_{i1} & s_{i2} & s_{i3}
\end{bmatrix}
=
\frac{
\begin{bmatrix}
q_i^\top k_1 &
q_i^\top k_2 &
q_i^\top k_3
\end{bmatrix}
}{
\sqrt{d_k}
}.
$$

The purpose of the scaling is to keep the dot product scores in a reasonable range before softmax.


In [8]:
d_k = q_i.shape[0]

scores = raw_scores / math.sqrt(d_k)

print("scaled scores:", scores)
print("the shape of scores:", scores.shape)


scaled scores: tensor([[0.7071, 0.0000, 0.7071]])
the shape of scores: torch.Size([1, 3])


## Step 4: Convert the scores into attention weights

For one fixed query $i$, softmax is applied over all of its scores:

$$
a_{ij}
=
\frac{\exp(s_{ij})}
{\sum_{m=1}^{n}\exp(s_{im})}.
$$

For our example:

$$
\begin{bmatrix}
s_{i1} & s_{i2} & s_{i3}
\end{bmatrix}
\quad
\xrightarrow{\mathrm{softmax}}
\quad
\begin{bmatrix}
a_{i1} & a_{i2} & a_{i3}
\end{bmatrix}.
$$

The attention weights satisfy:

$$
a_{i1}+a_{i2}+a_{i3}=1.
$$

In the code, the three key positions are in the last dimension, so we use:

```python
torch.softmax(scores, dim=-1)
```


In [9]:
weights = torch.softmax(scores, dim=-1)

print("attention weights:", weights)
print("sum of weights:   ", weights.sum(dim=-1))


attention weights: tensor([[0.4011, 0.1978, 0.4011]])
sum of weights:    tensor([1.])


## Step 5: Use the weights to combine the values

The lecture formula is:

$$
z_i
=
\sum_{j=1}^{n}a_{ij}v_j.
$$

For three values:

$$
z_i
=
a_{i1}v_1
+
a_{i2}v_2
+
a_{i3}v_3.
$$

Let:

$$
v_1
=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix},
\qquad
v_2
=
\begin{bmatrix}
0\\
1\\
0
\end{bmatrix},
\qquad
v_3
=
\begin{bmatrix}
0\\
0\\
1
\end{bmatrix}.
$$

Each value is a column vector:

$$
v_j\in\mathbb{R}^{3\times1}.
$$

Just as we did with the keys, place the values side by side:

$$
V_{\mathrm{cols}}
=
\begin{bmatrix}
| & | & |\\
v_1 & v_2 & v_3\\
| & | & |
\end{bmatrix}
=
\begin{bmatrix}
1 & 0 & 0\\
0 & 1 & 0\\
0 & 0 & 1
\end{bmatrix}
\in
\mathbb{R}^{3\times3}.
$$

The attention weights are currently stored as a row:

$$
a_i
=
\begin{bmatrix}
a_{i1} & a_{i2} & a_{i3}
\end{bmatrix}.
$$

So:

$$
a_i^\top
=
\begin{bmatrix}
a_{i1}\\
a_{i2}\\
a_{i3}
\end{bmatrix}.
$$

Now:

$$
z_i
=
V_{\mathrm{cols}}a_i^\top.
$$

Written out:

$$
z_i
=
\begin{bmatrix}
| & | & |\\
v_1 & v_2 & v_3\\
| & | & |
\end{bmatrix}
\begin{bmatrix}
a_{i1}\\
a_{i2}\\
a_{i3}
\end{bmatrix}
=
a_{i1}v_1+a_{i2}v_2+a_{i3}v_3.
$$

The shape is:

$$
(d_v\times n)(n\times1)
\rightarrow
(d_v\times1).
$$

So the result $z_i$ is again a **column vector**, exactly as in the lecture notation.


In [10]:
# Each column is one value vector v_j.
# Shape: (d_v, n) = (3, 3)
values_as_columns = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
])

# weights has shape (1, n), so weights.T has shape (n, 1).
# The result z_i is a column vector with shape (d_v, 1).
z_i = values_as_columns @ weights.T

print("values_as_columns shape:", values_as_columns.shape)
print("weights.T shape:         ", weights.T.shape)
print("z_i shape:               ", z_i.shape)
print("z_i:")
print(z_i)


values_as_columns shape: torch.Size([3, 3])
weights.T shape:          torch.Size([3, 1])
z_i shape:                torch.Size([3, 1])
z_i:
tensor([[0.4011],
        [0.1978],
        [0.4011]])


## Part 3 Summary

The whole numeric example follows the lecture notation directly.

### Query and Key matching

$$
q_i^\top
\begin{bmatrix}
k_1 & k_2 & \cdots & k_n
\end{bmatrix}
=
\begin{bmatrix}
q_i^\top k_1 &
q_i^\top k_2 &
\cdots &
q_i^\top k_n
\end{bmatrix}.
$$

Then scale:

$$
s_{ij}
=
\frac{q_i^\top k_j}{\sqrt{d_k}}.
$$

Then softmax:

$$
a_{ij}
=
\frac{\exp(s_{ij})}
{\sum_{m=1}^{n}\exp(s_{im})}.
$$

Finally combine the values:

$$
z_i
=
\sum_{j=1}^{n}a_{ij}v_j.
$$


## Important Notes

### 1. Why do we use this simple convention in Week 9?

The main purpose of Week 9 is to understand **how self attention works from the perspective of one token**.

For this reason, we deliberately use a simple representation:

* $q_i$ is kept as one column vector.
* All key vectors are placed side by side as columns.
* All value vectors are also placed side by side as columns.

This makes the calculation easy to see:

$$
q_i^\top
\begin{bmatrix}
k_1 & k_2 & \cdots & k_n
\end{bmatrix}
=
\begin{bmatrix}
q_i^\top k_1 &
q_i^\top k_2 &
\cdots &
q_i^\top k_n
\end{bmatrix}.
$$

The goal here is **not** to introduce the full matrix form of self attention yet.

The two main goals are:

1. Understand the self attention calculation from the perspective of **one token**.
2. Become familiar with using **PyTorch matrix operations** to implement the mathematical calculation.

The formal matrix representation for calculating attention for all tokens at once will be introduced in Week 10.

### 2. Where do the Query, Key, and Value numbers come from?

In this small example, values such as:

```text
[1, 0]
[0, 1]
[1, 1]
```

are simply **manually chosen example values**.

We use these simple numbers only to make the calculation easy to follow.

In a real self attention model, the query, key, and value vectors are not manually assigned.

They are produced from the input vector $x_i$ using learned projection matrices:

$$
q_i=W_Q^\top x_i,
$$

$$
k_i=W_K^\top x_i,
$$

$$
v_i=W_V^\top x_i.
$$

The matrices:

$$
W_Q,\qquad W_K,\qquad W_V
$$

are learned during model training.

Therefore, in a real model, the actual values of $q_i$, $k_i$, and $v_i$ depend on both:

* the input data $x_i$;
* the learned projection matrices $W_Q$, $W_K$, and $W_V$.

In this tutorial, we skip this projection step and directly provide simple Query, Key, and Value vectors so that we can focus on the **attention calculation itself**.

# Multiple Choice Questions

These questions are similar in level and style to the self attention questions you may meet in the exam.

Try each question first, then open the answer.

### Question 1

Suppose:

$$
d_k=16.
$$

What scaling divisor is used?

A. $16$  
B. $8$  
C. $4$  
D. $256$

<details>
<summary><b>Answer and explanation</b></summary>

**Answer: C**

Scaled dot product attention divides by:

$$
\sqrt{d_k}.
$$

Therefore:

$$
\sqrt{16}=4.
$$

</details>

### Question 2

For one fixed query $i$, which statement is correct?

A. The attention weights over all keys sum to 1.  
B. Every attention weight must be the same.  
C. The raw scores must sum to 1 before softmax.  
D. The keys are directly averaged to form the output.

<details>
<summary><b>Answer and explanation</b></summary>

**Answer: A**

Softmax produces one attention distribution over all key positions for the fixed query.

</details>

### Question 3

Which equation correctly describes the final attention output for token $i$?

A.

$$
z_i=\sum_{j=1}^{n}a_{ij}k_j
$$

B.

$$
z_i=\sum_{j=1}^{n}a_{ij}v_j
$$

C.

$$
z_i=q_i^\top k_i
$$

D.

$$
z_i=\sum_{j=1}^{n}v_j
$$

<details>
<summary><b>Answer and explanation</b></summary>

**Answer: B**

The attention weights are used to combine the **values**:

$$
z_i=\sum_{j=1}^{n}a_{ij}v_j.
$$

</details>
